# Student Academic Performance Segmentation

Unsupervised learning project: segment student cohorts using study hours, attendance rate, assignments submitted, and exam scores.

**Dataset note:** the included CSV is synthetic demonstration data, not actual student records. Use appropriately authorized, anonymized data for any real institutional analysis.

## 1. Imports and load data

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

from src.preprocessing import load_student_data, FEATURES
from src.clustering import evaluate_kmeans, fit_cluster_models
from src.visualization import plot_pca_clusters, plot_cluster_profiles

DATA_PATH = Path('data/student_performance.csv')
students = load_student_data(DATA_PATH)
print('Dataset shape:', students.shape)
display(students.head())
display(students[FEATURES].describe())

## 2. Explore features and scale before distance calculations
K-Means and Ward hierarchical clustering use distances, so feature scales matter. StandardScaler transforms each feature to approximately zero mean and unit variance. The scaler is fit once on this dataset for this unsupervised cohort exploration.

In [ ]:
display(students[FEATURES].hist(figsize=(10,7), bins=15))
plt.tight_layout(); plt.show()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(students[FEATURES])
print('Scaled feature means:', np.round(X_scaled.mean(axis=0), 3))
print('Scaled feature standard deviations:', np.round(X_scaled.std(axis=0), 3))

## 3. Select candidate cluster count: Elbow and Silhouette
The elbow plot shows within-cluster sum of squares (inertia); look for diminishing returns. The silhouette score measures how well-separated clusters are (higher is generally better). Neither metric alone proves a uniquely correct k, so combine them with interpretability and domain context.

In [ ]:
evaluation = pd.DataFrame(evaluate_kmeans(X_scaled, range(2, 8)))
display(evaluation)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(evaluation['k'], evaluation['inertia'], marker='o')
axes[0].set(title='Elbow Method', xlabel='Number of clusters (k)', ylabel='Inertia')
axes[1].plot(evaluation['k'], evaluation['silhouette'], marker='o')
axes[1].set(title='Silhouette Analysis', xlabel='Number of clusters (k)', ylabel='Silhouette score')
plt.tight_layout(); plt.show()

suggested_k = int(evaluation.loc[evaluation['silhouette'].idxmax(), 'k'])
print('Highest silhouette candidate k:', suggested_k)
# Review the elbow plot and persona usefulness before finalizing k.
FINAL_K = suggested_k

## 4. Fit K-Means and Hierarchical clustering
We compare both methods using the same scaled features and selected k. Silhouette is a useful internal comparison metric, not an external measure of educational success.

In [ ]:
kmeans, kmeans_labels, hierarchical_labels, scores = fit_cluster_models(X_scaled, FINAL_K)
print(scores)
students['kmeans_cluster'] = kmeans_labels
students['hierarchical_cluster'] = hierarchical_labels
display(students.head())

## 5. PCA 2D and 3D cluster visualizations

In [ ]:
fig, pca, coords = plot_pca_clusters(X_scaled, kmeans_labels)
plt.show()
print('PCA explained variance ratio:', np.round(pca.explained_variance_ratio_, 3))

pca3 = PCA(n_components=3)
coords3 = pca3.fit_transform(X_scaled)
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
sc = ax.scatter(coords3[:,0], coords3[:,1], coords3[:,2], c=kmeans_labels, cmap='viridis')
ax.set(title='Student Clusters (PCA 3D)', xlabel='PC1', ylabel='PC2', zlabel='PC3')
fig.colorbar(sc, ax=ax, label='Cluster', pad=0.1)
plt.tight_layout(); plt.show()

## 6. Cluster profiles and student personas
Persona labels below are descriptive summaries of the observed cluster averages, not fixed judgments about students. Interpret them with care and use them to guide supportive resources, not high-stakes decisions.

In [ ]:
fig, profile = plot_cluster_profiles(students, kmeans_labels)
plt.show()
display(profile.round(2))

overall = students[FEATURES].mean()
def describe_persona(cluster_id, row):
    study = 'higher' if row.study_hours >= overall.study_hours else 'lower'
    attendance = 'higher' if row.attendance_rate >= overall.attendance_rate else 'lower'
    submissions = 'more' if row.assignments_submitted >= overall.assignments_submitted else 'fewer'
    score = 'higher' if row.exam_score >= overall.exam_score else 'lower'
    return (f'Cluster {cluster_id}: {study} study time, {attendance} attendance, '
            f'{submissions} submitted assignments, and {score} average exam scores.')

for cluster_id, row in profile.iterrows():
    print(describe_persona(cluster_id, row))

## 7. Optional t-SNE visualization
t-SNE is useful for exploratory visualization but can distort global distances and is sensitive to settings. It is not used to fit the clustering model.

In [ ]:
from sklearn.manifold import TSNE
perplexity = min(30, max(2, (len(students)-1)//3))
tsne = TSNE(n_components=2, perplexity=perplexity, random_state=42, init='pca', learning_rate='auto')
tsne_coords = tsne.fit_transform(X_scaled)
plt.figure(figsize=(8,5))
plt.scatter(tsne_coords[:,0], tsne_coords[:,1], c=kmeans_labels, cmap='viridis', alpha=0.8)
plt.title('Student Clusters (t-SNE 2D)')
plt.xlabel('t-SNE 1'); plt.ylabel('t-SNE 2')
plt.tight_layout(); plt.show()

## 8. Export cluster assignments and summary
The exported file contains synthetic/demo student IDs. For real data, avoid exporting personally identifiable information and follow institutional privacy requirements.

In [ ]:
output = students[['student_id'] + FEATURES + ['kmeans_cluster', 'hierarchical_cluster']]
output.to_csv('student_cluster_assignments.csv', index=False)
profile.to_csv('student_cluster_profiles.csv')
print('Saved student_cluster_assignments.csv and student_cluster_profiles.csv')

## 9. Conclusions and limitations
- Standardization is important because distance-based clustering is sensitive to feature scale.
- Elbow and silhouette results are evidence to consider, not automatic proof of the best k.
- PCA/t-SNE are projections for visualization; cluster fitting used the original scaled features.
- Clusters are descriptive patterns, not diagnoses or measures of a student's potential.
- Synthetic demo data is only for testing the workflow. Validate any findings on representative, authorized data and consult educators before acting on them.